# 04 | Spatial crop mapping with WorldCereal 2026
Apply your newly trained classifier to the 2024 French spatial embeddings. Use the official cropland mask to restrict predictions, export georeferenced class and confidence rasters, and analyze mapped area. This is a demonstrator, not a validated crop inventory. Independent 2024 field labels are required for spatial accuracy assessment.

## 1. Load and validate inputs
Run after Notebooks 02 and 03. Both the embedding raster and classifier manifest must have the same checkpoint fingerprint and feature order. The cropland head is independent and may make errors.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/ESA_EO_Projects')
matches=[p.parent.parent for p in root.rglob('extractions.parquet') if (p.parent/'processed_training_df.parquet').is_file()]
assert len(matches)==1, matches
PROJECT=matches[0]; DATA=PROJECT/'data'; BASE=PROJECT/'outputs'; OLD=PROJECT/'models'
OUT=BASE/'2026'; MODELS=OLD/'2026'
for d in [OUT,MODELS]: d.mkdir(parents=True,exist_ok=True)
print(PROJECT)
import importlib.util, subprocess, sys
if importlib.util.find_spec('rioxarray') is None: subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rioxarray'])
import numpy as np,pandas as pd,xarray as xr,rioxarray,joblib,sklearn,matplotlib.pyplot as plt
from pyproj import CRS
from matplotlib.colors import ListedColormap,BoundaryNorm
from matplotlib.patches import Patch
m=json.loads((MODELS/'classifier_manifest.json').read_text())
assert sklearn.__version__==m['sklearn_version'], 'Match classifier scikit-learn version before loading'
rf=joblib.load(MODELS/'random_forest_2026.joblib')
e=xr.open_dataset(OUT/'spatial_embeddings_2024.nc',decode_coords='all')['presto_embedding']
mask=rioxarray.open_rasterio(OUT/'cropland_mask_2024.tif').squeeze('band',drop=True)
assert e.attrs['checkpoint_sha256']==m['checkpoint_sha256'] and e.feature.values.tolist()==m['features']
assert np.array_equal(e.x,mask.x) and np.array_equal(e.y,mask.y)
if e.rio.crs is None:
    with xr.open_dataset(DATA/'worldcereal-preprocessedinputs-french-demo.nc') as source:
        source_crs=CRS.from_wkt(source.crs.attrs['crs_wkt'])
        assert source_crs.to_epsg()==32631 and mask.rio.crs.to_epsg()==32631
        assert np.array_equal(e.x,source.x) and np.array_equal(e.y,source.y)
        e=e.rio.write_crs(source_crs)
assert e.rio.crs==mask.rio.crs and e.rio.transform()==mask.rio.transform()
assert e.attrs['pool_months']==','.join(map(str,m['pool_months']))
print('Grid:',e.shape,'CRS:',e.rio.crs,'Classes:',rf.classes_)

## 2. Predict crop classes and confidence
Use only valid cropland pixels. Code 0 represents non-cropland, 255 represents unknown/nodata, and 1–8 represent crop categories. Maximum class probability is a model confidence score, not a calibrated probability of being correct. Maintain the raster row-major orientation.

In [ ]:
v=e.transpose('y','x','feature').values; h,w=v.shape[:2]; maskv=mask.values
valid=np.isfinite(v).all(axis=-1)&(maskv==1)
assert valid.any(), 'No usable cropland pixels'
grid=np.full((h,w),255,dtype=np.uint8); grid[maskv==0]=0
confidence=np.full((h,w),np.nan,dtype=np.float32)
ids=np.flatnonzero(valid.ravel()); flat=v.reshape(-1,v.shape[-1]); classes=rf.classes_.tolist(); lookup={i+1:c for i,c in enumerate(classes)}
for k in range(0,len(ids),10000):
    ix=ids[k:k+10000]; proba=rf.predict_proba(pd.DataFrame(flat[ix],columns=m['features']))
    grid.ravel()[ix]=proba.argmax(axis=1)+1; confidence.ravel()[ix]=proba.max(axis=1)
print('Mapped crop pixels:',len(ids),'of',h*w); print(lookup)

## 3. Export GeoTIFF and legend
Keep the UTM projection and original pixel coordinates. Save discrete crop codes separately from continuous confidence. Calculate mapped area from pixel area, without interpreting it as unbiased reference area.

In [ ]:
def raster(a,name,nodata):
    q=xr.DataArray(a,dims=['y','x'],coords={'y':e.y,'x':e.x},name=name).rio.write_crs(e.rio.crs).rio.write_nodata(nodata)
    q.rio.to_raster(OUT/f'{name}.tif',compress='deflate')
raster(grid,'crop_type_2024',255); raster(confidence,'crop_confidence_2024',np.nan)
pixel_area=abs(float(np.median(np.diff(e.x)))*float(np.median(np.diff(e.y))))
legend=pd.DataFrame([{'code':0,'class':'non_cropland'},*({'code':k,'class':c} for k,c in lookup.items()),{'code':255,'class':'nodata'}]); legend['pixels']=legend.code.map(lambda k:int((grid==k).sum())); legend['mapped_km2']=legend.pixels*pixel_area/1e6
legend.to_csv(OUT/'crop_legend_and_area.csv',index=False); display(legend)

## 4. Visualize the crop map and confidence
Map colors are categorical, not a continuous spectral scale. The confidence panel helps identify ambiguous regions. Export a portfolio figure; use the GeoTIFFs in QGIS.

In [ ]:
colors=['#e0e0e0','#e69f00','#56b4e9','#009e73','#f0e442','#0072b2','#d55e00','#cc79a7','#8c564b']
cmap=ListedColormap(colors); cmap.set_bad('white'); norm=BoundaryNorm(np.arange(-.5,9.5),cmap.N)
fig,ax=plt.subplots(1,2,figsize=(15,7),constrained_layout=True)
ax[0].imshow(np.ma.masked_equal(grid,255),cmap=cmap,norm=norm); ax[0].set_title('Predicted crop types | 2024')
ax[0].legend(handles=[Patch(color=colors[k],label=c.replace('_',' ')) for k,c in {0:'non_cropland',**lookup}.items()],loc='upper center',bbox_to_anchor=(.5,-.03),ncol=2,fontsize=8)
im=ax[1].imshow(confidence,vmin=0,vmax=1,cmap='viridis'); ax[1].set_title('Maximum class probability'); fig.colorbar(im,ax=ax[1],shrink=.75)
for a in ax: a.axis('off')
fig.savefig(OUT/'crop_mapping_overview.png',dpi=200,bbox_inches='tight'); plt.show()

## 5. Research reflection
Record retained training/test sample numbers, matched comparison metrics, minority crop performance, cropland pixel count and mapped crop areas. Avoid reporting held-out sample accuracy as map accuracy. A rigorous next experiment would use 2024 independent parcel labels, assess spatial overlap and test cross-country transferability.